In [2]:
%pip install pandas

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   --------------------------------- ------ 8.1/9.6 MB 47.1 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 37.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import re
import html
import unicodedata
import pandas as pd

def clean_financial_text(text: str) -> str:
    # 1. HTML 实体解码 (例如 & euro ; -> €)
    text = html.unescape(text)
    
    # 2. Unicode 标准化，消除组合字符异常
    text = unicodedata.normalize('NFKD', text)
    
    # 3. 修复损坏的特殊字符与多余连字符
    text = re.sub(r'\+[\w]+\+[\w]+', '', text)  # 去除乱码占位符
    text = text.replace('+ó', '').replace('+-', '').replace('-LR', '')
    
    # 4. 规范化金融单位与浮点数值间距 (例如 EUR 100 . 5 m -> EUR 100.5m)
    text = re.sub(r'(\d+)\s*\.\s*(\d+)', r'\1.\2', text)
    text = re.sub(r'(\d+)\s*%', r'\1%', text)
    text = re.sub(r'(EUR|USD|GBP|SEK|NOK)\s*(\d+)', r'\1 \2', text)
    
    # 5. 去除多余括号及多余空白字符
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_and_clean_file(file_path: str) -> pd.DataFrame:
    records = []
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or "@" not in line:
                continue
            
            # 依最右侧 @ 拆分句子与标签
            sentence, label = line.rsplit("@", 1)
            clean_s = clean_financial_text(sentence)
            label = label.strip().lower()
            
            if label in ["positive", "neutral", "negative"] and len(clean_s) > 5:
                records.append({"text": clean_s, "label": label})
                
    df = pd.DataFrame(records)
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    df["label_id"] = df["label"].map(label_map)
    return df

# 使用示例：
# df = parse_and_clean_file("Sentences_50Agree.txt")
# print(df["label"].value_counts())

In [4]:
import os
import csv

# 打印文件完整路径
csv_path = "cleaned_data.csv"
print("文件完整绝对路径:", os.path.abspath(csv_path))

# 读取前 5 条清洗结果查看
with open(csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 5:
            break
        print(f"\n[样本 {i+1}]")
        print(f"文本: {row['text']}")
        print(f"标签: {row['label']} (ID: {row['label_id']})")

文件完整绝对路径: C:\Users\clark\Desktop\project for cv\20\FinancialPhraseBank-v1.0\FinancialPhraseBank-v1.0\cleaned_data.csv


FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_data.csv'

In [5]:
import os

print("当前工作目录:", os.getcwd())
print("当前目录下的文件与文件夹:")
for f in os.listdir("."):
    if f.endswith(".txt") or f.endswith(".csv"):
        print("  -", f)

当前工作目录: C:\Users\clark\Desktop\project for cv\20\FinancialPhraseBank-v1.0\FinancialPhraseBank-v1.0
当前目录下的文件与文件夹:
  - License.txt
  - README.txt
  - Sentences_50Agree.txt
  - Sentences_66Agree.txt
  - Sentences_75Agree.txt
  - Sentences_AllAgree.txt


In [6]:
import csv
import html
import os
import re
import unicodedata

# 1. 设置输入与输出路径
# 如果 Sentences_50Agree.txt 在子目录，请替换为它的真实路径（如 "data/Sentences_50Agree.txt"）
input_file = "Sentences_50Agree.txt"
output_file = "cleaned_data.csv"

# 检查输入源文件是否存在
if not os.path.exists(input_file):
    print(f"❌ 找不到输入文件: {os.path.abspath(input_file)}")
    print("👉 请检查文件名是否准确，或者将 input_file 变量修改为完整绝对路径。")
else:
    def clean_financial_text(text: str) -> str:
        text = html.unescape(text)
        text = unicodedata.normalize('NFKD', text)
        text = re.sub(r'\+[\w]+\+[\w]+', '', text)
        text = text.replace('+ó', '').replace('+-', '').replace('-LR', '')
        text = re.sub(r'(\d+)\s*\.\s*(\d+)', r'\1.\2', text)
        text = re.sub(r'(\d+)\s*%', r'\1%', text)
        text = re.sub(r'(EUR|USD|GBP|SEK|NOK)\s*(\d+)', r'\1 \2', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    records = []

    with open(input_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or "@" not in line:
                continue
            sentence, label = line.rsplit("@", 1)
            clean_s = clean_financial_text(sentence)
            lbl = label.strip().lower()
            if lbl in label_map and len(clean_s) > 5:
                records.append({
                    "text": clean_s,
                    "label": lbl,
                    "label_id": label_map[lbl]
                })

    # 写入 CSV
    with open(output_file, "w", encoding="utf-8", newline="") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=["text", "label", "label_id"])
        writer.writeheader()
        writer.writerows(records)

    abs_out_path = os.path.abspath(output_file)
    print(f"✅ 清洗成功！共处理 {len(records)} 条数据")
    print(f"📁 文件已写入保存至: {abs_out_path}")

✅ 清洗成功！共处理 4846 条数据
📁 文件已写入保存至: C:\Users\clark\Desktop\project for cv\20\FinancialPhraseBank-v1.0\FinancialPhraseBank-v1.0\cleaned_data.csv


In [7]:
pip install torch transformers datasets scikit-learn pandas numpy

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------  12.3/12.6 MB 66.0 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 58.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 31.7 MB/s  0:00:00
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ---------------------------------------- 846.4/846.4 kB 53.7 MB/s  0:00:00
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 28.1 MB/s  0:00:00
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   ----------------- ---------------------- 12.3/27.9 MB 61.3 MB/s eta 0:00:01
   ---------------------------------- ----- 24.4/27.9 MB 60.8 MB/s eta 0:00:01
   ---------------------------------------- 27.9/27.9 MB 56.3 MB/s  0:00:00

   ---- ---------------------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# ==========================================
# 1. 超参数与设备配置
# ==========================================
MODEL_NAME = "ProsusAI/finbert"  # 经典开源 FinBERT 权重
DATA_PATH = "cleaned_data.csv"
BATCH_SIZE = 16
MAX_LEN = 128
EPOCHS = 4
LR = 2e-5
SEED = 42

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用计算设备: {device}")

# ==========================================
# 2. 数据读取与分层划分 (Stratified Split)
# ==========================================
df = pd.read_csv(DATA_PATH)
df['text'] = df['text'].astype(str)
df['label_id'] = df['label_id'].astype(int)

# 70% 训练, 15% 验证, 15% 测试
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df['label_id']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label_id']
)

print(f"训练集样本数: {len(train_df)}")
print(f"验证集样本数: {len(val_df)}")
print(f"测试集样本数: {len(test_df)}")

# 计算类别权重以应对不平衡 (Negative 占比低)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df['label_id']),
    y=train_df['label_id'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"各类别损失权重 [Negative, Neutral, Positive]: {class_weights_tensor.tolist()}")

# ==========================================
# 3. 自定义 Dataset 与 DataLoader
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FinancialPhraseDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = FinancialPhraseDataset(train_df['text'].tolist(), train_df['label_id'].tolist(), tokenizer, MAX_LEN)
val_dataset = FinancialPhraseDataset(val_df['text'].tolist(), val_df['label_id'].tolist(), tokenizer, MAX_LEN)
test_dataset = FinancialPhraseDataset(test_df['text'].tolist(), test_df['label_id'].tolist(), tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. 初始化模型、损失函数与优化器
# ==========================================
# 映射: 0 -> negative, 1 -> neutral, 2 -> positive
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True
)
model.to(device)

# 带权重的交叉熵损失
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

# ==========================================
# 5. 训练与验证循环
# ==========================================
def evaluate(model, data_loader):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, macro_f1, all_preds, all_labels

print("\n--- 开始微调 FinBERT ---")
best_macro_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        torch.save(model.state_dict(), "best_finbert_model.pt")
        print(" -> 验证集 Macro-F1 创新高，保存最佳模型权重。")

# ==========================================
# 6. 测试集最终评估
# ==========================================
print("\n--- 正在评估测试集表现 ---")
model.load_state_dict(torch.load("best_finbert_model.pt"))
test_loss, test_acc, test_macro_f1, test_preds, test_labels = evaluate(model, test_loader)

target_names = ["negative", "neutral", "positive"]
print(f"测试集 Accuracy: {test_acc:.4f}")
print(f"测试集 Macro-F1: {test_macro_f1:.4f}")
print("\n分类指标明细报告:")
print(classification_report(test_labels, test_preds, target_names=target_names, digits=4))

print("混淆矩阵:")
cm = confusion_matrix(test_labels, test_preds)
cm_df = pd.DataFrame(cm, index=[f"真实_{name}" for name in target_names], columns=[f"预测_{name}" for name in target_names])
print(cm_df)

C:\Users\clark\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


使用计算设备: cpu
训练集样本数: 3392
验证集样本数: 727
测试集样本数: 727
各类别损失权重 [Negative, Neutral, Positive]: [2.672970771789551, 0.5611249208450317, 1.185185194015503]


C:\Users\clark\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\clark\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\clark\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_dow


--- 开始微调 FinBERT ---


KeyboardInterrupt: 

In [9]:
from tqdm import tqdm
import time

# ====================================================
# CPU 建议加速参数调整（可显著缩短 CPU 耗时）：
# 1. MAX_LEN: 从 128 减到 64（新闻句子一般较短，计算量减少近半）
# 2. EPOCHS: CPU 微调建议先设为 2~3 轮
# ====================================================

print("\n--- 开始微调 FinBERT ---")
best_macro_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    start_time = time.time()

    # 使用 tqdm 包装 train_loader，实时查看每个 batch 的耗时和预计剩余时间
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for step, batch in enumerate(train_bar):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        
        # 实时刷新进度条上的 loss
        train_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)
    
    # 验证集评估
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader)
    elapsed = time.time() - start_time

    print(f"\n✅ Epoch {epoch+1}/{EPOCHS} 完成 (耗时: {elapsed:.1f}s) | "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        torch.save(model.state_dict(), "best_finbert_model.pt")
        print("  🌟 验证集 Macro-F1 创新高，最佳权重已保存。")


--- 开始微调 FinBERT ---


Epoch 1/4 [Train]:  87%|████████████████████████████████████▋     | 185/212 [15:27<02:15,  5.01s/it, batch_loss=0.4988]


KeyboardInterrupt: 

In [ ]:
import torch
print("CUDA 是否可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("显卡型号:", torch.cuda.get_device_name(0))